# Is Agency Worth the Tokens? Cost–Accuracy Trade-offs of Agentic Scaffolds for Small Language Models

**Research questions**
- **RQ1** Does giving a small LM a code tool (program-aided reasoning) beat chain-of-thought (CoT)?
- **RQ2** Does an *agentic* execute→observe→repair loop add accuracy, and at what token cost?
- **RQ3** Is it better to spend tokens on repair (sequential) or on sampling + voting (parallel)? Where is the Pareto frontier?

**Conditions:** `direct`, `cot`, `pal` (tool, single shot), `pal_repair` (agent loop with execution feedback), `cot_sc` (CoT, K samples, vote), `pal_sc` (tool, K samples, vote).

**Data:** GSM8K test split, downloaded through the **kagglehub API** (`thedevastator/grade-school-math-8k-q-a`).

**How to run (Colab):** Runtime → *T4 GPU* → Run all.
Add `KAGGLE_USERNAME` and `KAGGLE_KEY` in Colab *Secrets* (left sidebar, key icon), or you will be prompted by `kagglehub.login()`.
Set `SMOKE = True` first for a 5-minute test. Re-run with other `MODEL_ID`s (e.g. `Qwen/Qwen2.5-0.5B-Instruct`, `Qwen/Qwen2.5-3B-Instruct`) to get a model-scale axis; the analysis cell merges all models automatically.

In [ ]:
# Do NOT use -U on pandas/scipy/matplotlib: Colab pins its own versions.
!pip -q install pandas==2.2.3
!pip -q install kagglehub "transformers>=4.44" accelerate

In [ ]:
import os, re, sys, json, time, random, subprocess, tempfile, glob, collections, math, resource
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch

# ------------------------- CONFIG -------------------------
MODELS      = ["Qwen/Qwen2.5-0.5B-Instruct", "Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct"]  # run in order; checkpoints let you resume
N_EVAL      = 500      # GSM8K test questions (fixed random subset). Rough T4 time per 500 Qs: 0.5B ~25 min, 1.5B ~55 min, 3B ~2 h
K_SAMPLES   = 5        # samples for self-consistency conditions
TEMPERATURE = 0.7
MAX_REPAIR  = 2        # max repair rounds in the agent loop
MAX_NEW_COT = 512    # raised from 384: 12% of CoT runs were truncated at 384
MAX_NEW_PAL = 320
BATCH       = 16
SEED        = 42
SMOKE       = False    # True = tiny run to test the pipeline
# ----------------------------------------------------------
if SMOKE: N_EVAL, K_SAMPLES = 24, 3
# results go in a folder named by the settings, so runs with different settings never overwrite each other
OUT_DIR = f"/content/results/n{N_EVAL}_cot{MAX_NEW_COT}"; os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
import kagglehub
try:
    from google.colab import userdata
    os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
except Exception as e:
    print("Colab secrets not found -> interactive login")
    kagglehub.login()

path = kagglehub.dataset_download("thedevastator/grade-school-math-8k-q-a")
csvs = glob.glob(os.path.join(path, "**", "*.csv"), recursive=True)
print(csvs)
test_csv = next(p for p in csvs if "main_test" in os.path.basename(p))
df = pd.read_csv(test_csv)
assert {"question", "answer"} <= set(df.columns), df.columns
df["gold"] = df["answer"].str.split("####").str[-1].str.strip().str.replace(",", "").astype(float)
data = df.sample(N_EVAL, random_state=SEED).reset_index(drop=True)
print(len(df), "test rows ->", len(data), "evaluated")
data.head(3)

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
def load_model(model_id):
    global tok, model
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto").eval()
    print("loaded", model_id)

In [ ]:
# ---------------------- generation ----------------------
@torch.no_grad()
def generate(chats, max_new_tokens, temperature=0.0, n=1):
    """chats: list of message lists. Returns, per chat, a dict with n texts, n generated-token
    counts and the prompt-token count (prompt counted ONCE per call, as with API 'n' sampling)."""
    prompts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True) for c in chats]
    out = [dict(texts=[], gen_tokens=[], prompt_tokens=0) for _ in chats]
    sample = temperature > 0
    for s in range(0, len(prompts), BATCH):
        enc = tok(prompts[s:s + BATCH], return_tensors="pt", padding=True).to(model.device)
        kw = dict(max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id)
        if sample: kw.update(do_sample=True, temperature=temperature, top_p=0.95, num_return_sequences=n)
        else:      kw.update(do_sample=False, temperature=None, top_p=None, top_k=None)
        g = model.generate(**enc, **kw)[:, enc["input_ids"].shape[1]:]
        per = n if sample else 1
        for j in range(g.shape[0]):
            i = s + j // per
            out[i]["texts"].append(tok.decode(g[j], skip_special_tokens=True))
            out[i]["gen_tokens"].append(int((g[j] != tok.pad_token_id).sum()))
        for b in range(enc["input_ids"].shape[0]):
            out[s + b]["prompt_tokens"] = int(enc["attention_mask"][b].sum())
    return out

# ---------------------- answer parsing ----------------------
NUM = re.compile(r"-?\d[\d,]*\.?\d*")
def to_num(s):
    try:
        v = float(str(s).replace(",", "").rstrip("."))
        return v if math.isfinite(v) else None
    except Exception:
        return None

def extract_final(text):
    m = re.findall(r"answer is[:\s\$]*(-?\d[\d,]*\.?\d*)", text, flags=re.I)
    if m: return to_num(m[-1])
    m = re.findall(r"\\boxed\{([^}]*)\}", text)
    if m:
        n = NUM.findall(m[-1])
        if n: return to_num(n[-1])
    n = NUM.findall(text)
    return to_num(n[-1]) if n else None

def is_correct(pred, gold): return pred is not None and abs(pred - gold) < 1e-4

# ---------------------- tool: sandboxed python ----------------------
FENCE = "`" * 3
def extract_code(text):
    m = re.search(FENCE + r"(?:python)?\s*\n(.*?)(?:" + FENCE + r"|$)", text, flags=re.S)
    return (m.group(1) if m else text).strip()

def _limits():
    resource.setrlimit(resource.RLIMIT_AS, (2 * 1024**3, 2 * 1024**3))

def run_code(code, timeout=5):
    """Run model-written code in a separate, resource-limited interpreter. Returns (value, error)."""
    prog = code + "\n\ntry:\n    _r = solve()\n    print('__RESULT__', _r)\nexcept Exception as e:\n    print('__ERROR__', repr(e))\n"
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(prog); fn = f.name
    try:
        p = subprocess.run([sys.executable, "-I", fn], capture_output=True, text=True,
                           timeout=timeout, preexec_fn=_limits)
    except subprocess.TimeoutExpired:
        return None, "TimeoutError: program exceeded time limit"
    finally:
        try: os.remove(fn)
        except OSError: pass
    lines = [l for l in p.stdout.splitlines() if l.startswith("__RESULT__") or l.startswith("__ERROR__")]
    if lines and lines[-1].startswith("__RESULT__"):
        val = to_num(lines[-1][len("__RESULT__"):].strip().split()[0]) if lines[-1][len("__RESULT__"):].strip() else None
        return (val, None) if val is not None else (None, "ValueError: solve() did not return a finite number")
    if lines: return None, lines[-1][len("__ERROR__"):].strip()
    err = (p.stderr.strip().splitlines() or ["no output (is `solve` defined?)"])[-1]
    return None, err

def exec_many(codes):
    with ThreadPoolExecutor(8) as ex:
        return list(ex.map(run_code, codes))

def vote(vals):
    vals = [round(v, 4) for v in vals if v is not None]
    return collections.Counter(vals).most_common(1)[0][0] if vals else None

In [ ]:
# ---------------------- prompts ----------------------
SYS = "You are a careful math problem solver."
def chat(user): return [{"role": "system", "content": SYS}, {"role": "user", "content": user}]
P_DIRECT = "{q}\n\nRespond with only the final number in the form: The answer is <number>."
P_COT    = "{q}\n\nThink step by step, then finish with: The answer is <number>."
P_PAL    = ("{q}\n\nWrite a Python function `solve()` that returns the final numeric answer. "
            "Use plain Python only (you may import math or fractions). Do not read input. "
            "Reply with a single " + FENCE + "python code block and nothing else.")

def rec(i, pred, gen, prm, **extra):
    return dict(qid=i, gold=float(data.gold[i]), pred=pred, correct=bool(is_correct(pred, data.gold[i])),
                gen_tokens=int(gen), prompt_tokens=int(prm), total_tokens=int(gen + prm), **extra)

# ---------------------- conditions ----------------------
def cond_text(tpl, max_new, n=1, temp=0.0):
    outs = generate([chat(tpl.format(q=q)) for q in data.question], max_new, temp, n)
    recs = []
    for i, o in enumerate(outs):
        preds = [extract_final(t) for t in o["texts"]]
        pred = preds[0] if n == 1 else vote(preds)
        extra = {} if n == 1 else dict(votes=preds, oracle=any(is_correct(p, data.gold[i]) for p in preds))
        recs.append(rec(i, pred, sum(o["gen_tokens"]), o["prompt_tokens"], calls=n, **extra))
    return recs

def cond_pal():
    outs = generate([chat(P_PAL.format(q=q)) for q in data.question], MAX_NEW_PAL)
    res = exec_many([extract_code(o["texts"][0]) for o in outs])
    return [rec(i, v, o["gen_tokens"][0], o["prompt_tokens"], calls=1, text=o["texts"][0], val=v, err=e)
            for i, (o, (v, e)) in enumerate(zip(outs, res))]

def cond_pal_repair(pal_recs):
    """Agent loop: execute -> observe error -> repair, up to MAX_REPAIR rounds. Reuses the (greedy,
    deterministic) first attempt from `pal`, so its cost is carried over, not double counted."""
    S = [dict(r) for r in pal_recs]
    for s in S: s["repair_rounds"] = 0
    for rnd in range(MAX_REPAIR):
        todo = [i for i, s in enumerate(S) if s["val"] is None]
        if not todo: break
        chats = [chat(P_PAL.format(q=data.question[i])) + [
                    {"role": "assistant", "content": S[i]["text"]},
                    {"role": "user", "content": f"Running your code failed: {str(S[i]['err'])[:300]}\n"
                     f"Fix the bug and reply with the full corrected code in a single python block."}]
                 for i in todo]
        outs = generate(chats, MAX_NEW_PAL)
        res = exec_many([extract_code(o["texts"][0]) for o in outs])
        for i, o, (v, e) in zip(todo, outs, res):
            s = S[i]
            s.update(text=o["texts"][0], val=v, err=e, pred=v, repair_rounds=s["repair_rounds"] + 1,
                     calls=s["calls"] + 1, gen_tokens=s["gen_tokens"] + o["gen_tokens"][0],
                     prompt_tokens=s["prompt_tokens"] + o["prompt_tokens"])
            s["correct"] = bool(is_correct(v, data.gold[i])); s["total_tokens"] = s["gen_tokens"] + s["prompt_tokens"]
    return S

def cond_pal_sc():
    outs = generate([chat(P_PAL.format(q=q)) for q in data.question], MAX_NEW_PAL, TEMPERATURE, K_SAMPLES)
    flat = [extract_code(t) for o in outs for t in o["texts"]]
    res = exec_many(flat); recs = []
    for i, o in enumerate(outs):
        vals = [v for v, _ in res[i * K_SAMPLES:(i + 1) * K_SAMPLES]]
        recs.append(rec(i, vote(vals), sum(o["gen_tokens"]), o["prompt_tokens"], calls=K_SAMPLES, votes=vals,
                        oracle=any(is_correct(v, data.gold[i]) for v in vals), val=vote(vals)))
    return recs

In [ ]:
def ckpt(cond): return f"{OUT_DIR}/{MODEL_SLUG}__{cond}.json"
def load(cond): return json.load(open(ckpt(cond)))
def run(cond, fn):
    if os.path.exists(ckpt(cond)):
        print(f"[skip] {cond} (checkpoint exists)"); return load(cond)
    torch.manual_seed(SEED); t0 = time.time(); r = fn(); dt = time.time() - t0
    json.dump(r, open(ckpt(cond), "w"))
    acc = np.mean([x["correct"] for x in r]); tok_ = np.mean([x["total_tokens"] for x in r])
    print(f"[done] {cond:11s} acc={acc:.3f}  mean_tokens={tok_:7.1f}  time={dt/60:.1f} min")
    return r

for MODEL_ID in MODELS:
    MODEL_SLUG = MODEL_ID.split("/")[-1]
    print("=" * 10, MODEL_ID)
    load_model(MODEL_ID)
    R = {}
    R["direct"]     = run("direct",     lambda: cond_text(P_DIRECT, 24))
    R["cot"]        = run("cot",        lambda: cond_text(P_COT, MAX_NEW_COT))
    R["pal"]        = run("pal",        cond_pal)
    R["pal_repair"] = run("pal_repair", lambda: cond_pal_repair(R["pal"]))
    R["cot_sc"]     = run("cot_sc",     lambda: cond_text(P_COT, MAX_NEW_COT, K_SAMPLES, TEMPERATURE))
    R["pal_sc"]     = run("pal_sc",     cond_pal_sc)
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ---------------------- analysis (merges every model found in OUT_DIR) ----------------------
import matplotlib.pyplot as plt
from scipy.stats import binomtest

rows = []
for f in glob.glob(f"{OUT_DIR}/*__*.json"):
    model, cond = os.path.basename(f)[:-5].split("__")
    for r in json.load(open(f)): r.update(model=model, cond=cond); rows.append(r)
A = pd.DataFrame(rows)

# EXPLORATORY condition (derived from logged runs, no extra generation): accept the answer when the CoT and
# PAL answers agree, otherwise escalate to CoT-SC. It was suggested by a pilot on 300 questions, so it is
# a hypothesis-generating result here, not a pre-planned one. Report it as exploratory.
gate_rows = []
for model, g in A.groupby("model"):
    P = {c: g[g.cond == c].sort_values("qid").reset_index(drop=True) for c in ["cot", "pal", "cot_sc"]}
    if all(len(v) for v in P.values()):
        agree = ((P["cot"].pred.round(4) == P["pal"].pred.round(4)) & P["pal"].pred.notna()).values
        esc = ~agree
        for k in range(len(agree)):
            gt = P["cot"].gen_tokens[k] + P["pal"].gen_tokens[k] + (P["cot_sc"].gen_tokens[k] if esc[k] else 0)
            tt = P["cot"].total_tokens[k] + P["pal"].total_tokens[k] + (P["cot_sc"].total_tokens[k] if esc[k] else 0)
            gate_rows.append(dict(qid=int(P["cot"].qid[k]), model=model, cond="gate", calls=2 + (K_SAMPLES if esc[k] else 0),
                                  correct=bool(P["cot_sc"].correct[k] if esc[k] else P["cot"].correct[k]),
                                  gen_tokens=int(gt), total_tokens=int(tt)))
if gate_rows: A = pd.concat([A, pd.DataFrame(gate_rows)], ignore_index=True)

ORDER  = ["direct", "cot", "pal", "pal_repair", "cot_sc", "pal_sc", "gate"]
LABELS = dict(direct="Direct", cot="CoT", pal="PAL (tool)", pal_repair="PAL + repair (agent)",
              cot_sc=f"CoT-SC (k={K_SAMPLES})", pal_sc=f"PAL-SC (k={K_SAMPLES})", gate="Agree-gate (exploratory)")

def boot_ci(x, B=2000, seed=0):
    rng = np.random.default_rng(seed); x = np.asarray(x, float)
    m = [x[rng.integers(0, len(x), len(x))].mean() for _ in range(B)]
    return np.percentile(m, [2.5, 97.5])

tabs = []
for (model, cond), g in A.groupby(["model", "cond"]):
    lo, hi = boot_ci(g.correct)
    t = dict(model=model, cond=cond, n=len(g), acc=g.correct.mean(), ci_lo=lo, ci_hi=hi,
             gen_tok=g.gen_tokens.mean(), total_tok=g.total_tokens.mean(),
             acc_per_1k_tok=g.correct.mean() / (g.total_tokens.mean() / 1000), calls=g.calls.mean())
    if "val" in g: t["tool_fail_rate"] = g["val"].isna().mean() if cond in ("pal", "pal_repair") else np.nan
    if "oracle" in g and g.oracle.notna().any(): t["oracle_pass@k"] = g.oracle.mean()
    tabs.append(t)
T = pd.DataFrame(tabs)
T["cond"] = pd.Categorical(T.cond, ORDER, ordered=True); T = T.sort_values(["model", "cond"]).reset_index(drop=True)
pd.set_option("display.width", 200); pd.set_option("display.float_format", "{:.3f}".format)
print(T.to_string(index=False))
T.to_csv(f"{OUT_DIR}/summary_table.csv", index=False)

# paired exact McNemar tests
def mcnemar(a, b):
    b01 = int(((~a) & b).sum()); b10 = int((a & (~b)).sum()); n = b01 + b10
    return (1.0 if n == 0 else binomtest(b01, n, 0.5).pvalue), b10, b01
PAIRS = [("cot", "direct"), ("pal", "cot"), ("pal_repair", "pal"), ("pal_sc", "pal"),
         ("cot_sc", "cot"), ("pal_sc", "cot_sc"), ("pal_sc", "pal_repair"), ("gate", "cot_sc"), ("gate", "cot")]
tests = []
for model, g in A.groupby("model"):
    P = g.pivot(index="qid", columns="cond", values="correct")
    for x, y in PAIRS:
        if x in P and y in P:
            p, only_x, only_y = mcnemar(P[x].astype(bool), P[y].astype(bool))
            tests.append(dict(model=model, A=x, B=y, acc_diff=P[x].mean() - P[y].mean(),
                              only_A_right=only_x, only_B_right=only_y, p_mcnemar=p))
M = pd.DataFrame(tests); print("\n", M.to_string(index=False)); M.to_csv(f"{OUT_DIR}/paired_tests.csv", index=False)

# Pareto plot
models = sorted(A.model.unique())
fig, axes = plt.subplots(1, len(models), figsize=(5.2 * len(models), 4.2), squeeze=False)
for ax, model in zip(axes[0], models):
    s = T[T.model == model]
    for k, (_, r) in enumerate(s.iterrows()):
        ax.errorbar(r.total_tok, r.acc, yerr=[[r.acc - r.ci_lo], [r.ci_hi - r.acc]], fmt="o", capsize=3)
        ax.annotate(LABELS[r.cond], (r.total_tok, r.acc), textcoords="offset points", xytext=(6, 6 if k % 2 == 0 else -14), fontsize=8)
    ax.set_xscale("log"); ax.set_xlabel("mean tokens / question (prompt + generated, log)"); ax.set_ylabel("accuracy")
    ax.set_title(model); ax.grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/pareto.png", dpi=200); plt.show()

# LaTeX table for the paper
L = T.copy(); L["cond"] = L.cond.map(LABELS)
L["Accuracy [95% CI]"] = L.apply(lambda r: f"{100*r.acc:.1f} [{100*r.ci_lo:.1f}, {100*r.ci_hi:.1f}]", axis=1)
L = L[["model", "cond", "Accuracy [95% CI]", "total_tok", "acc_per_1k_tok"]]
L.columns = ["Model", "Method", "Acc. % [95% CI]", "Tokens/Q", "Acc./1k tok"]
open(f"{OUT_DIR}/results_table.tex", "w").write(L.to_latex(index=False, float_format="%.2f"))
print("\nSaved: summary_table.csv, paired_tests.csv, pareto.png, results_table.tex")

In [ ]:
# ---------------------- error analysis (feeds the Discussion section) ----------------------
for model, g in A.groupby("model"):
    print("=" * 20, model)
    pal, rep = g[g.cond == "pal"].set_index("qid"), g[g.cond == "pal_repair"].set_index("qid")
    if len(pal) and len(rep):
        failed = pal.val.isna()
        print(f"PAL tool-failure rate: {failed.mean():.3f}  -> after repair: {rep.val.isna().mean():.3f}")
        print(f"Repair recovered {int((failed & rep.val.notna()).sum())} / {int(failed.sum())} failed runs; "
              f"of those, correct: {int((failed & rep.correct).sum())}")
        silent = (~failed) & (~pal.correct)
        print(f"'Silent' wrong answers (ran fine, wrong value; repair cannot fix): {silent.mean():.3f}")
    cot_ = g[g.cond == "cot"]
    if len(cot_): print(f"CoT runs truncated at the {MAX_NEW_COT}-token cap: {(cot_.gen_tokens >= MAX_NEW_COT - 2).mean():.3f}")
    sc = g[g.cond == "pal_sc"]
    if len(sc):
        print(f"PAL-SC majority-vote acc {sc.correct.mean():.3f} vs oracle pass@{K_SAMPLES} {sc.oracle.mean():.3f} "
              f"(headroom for a better verifier: {sc.oracle.mean() - sc.correct.mean():.3f})")

# bundle everything for download
!cd /content && zip -qr results_bundle.zip results
try:
    from google.colab import files; files.download("/content/results_bundle.zip")
except Exception: pass

## Reproducibility notes (copy into the paper's appendix)
- Fixed seed, fixed random subset of GSM8K test (`N_EVAL`), greedy decoding for single-shot/agent runs, `T=0.7, top_p=0.95` for self-consistency.
- Token cost = prompt tokens + generated tokens; for K-sample conditions the prompt is counted once (API-style `n` sampling).
- Generated code executes in a separate interpreter with a 5 s timeout and 2 GB memory cap, inside the Colab VM. Do not run this on a machine holding secrets.
- Report the exact library versions printed by `pip freeze | grep -E "transformers|torch|kagglehub"` in the paper.
- GSM8K is MIT-licensed; cite Cobbe et al. (2021). The Kaggle copy is a mirror of the Hugging Face release; credit both.